[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 100. The Closed-Loop Network Design Problem
## Tier 6: The Autonomous Self-Optimizing Ecosystem

### Key Assumptions
- Multi-agent systems can autonomously optimize network operations
- Contract Net Protocol enables efficient agent negotiations
- Market-based coordination emerges optimal resource allocation
- Self-organizing systems adapt to changing conditions
- Emergent intelligence improves system performance over time
- Autonomous agents require minimal human intervention

### Approach (Step-by-Step)
1. **Multi-Agent Architecture**: Design autonomous agents for network components
2. **Contract Net Protocol**: Implement agent negotiation and task allocation
3. **Market-Based Coordination**: Create pricing mechanisms for resource allocation
4. **Self-Organization**: Enable agents to adapt and optimize autonomously
5. **Emergent Behavior**: Analyze system-level patterns from agent interactions
6. **Performance Evaluation**: Measure ecosystem effectiveness and resilience

### What to Look for in Results
- Multi-agent system architecture and agent behaviors
- Contract Net Protocol negotiation outcomes
- Market-based resource allocation efficiency
- Self-organizing network adaptation capabilities
- Emergent intelligence and system resilience
- Performance comparison with centralized approaches

### Concrete Example
We will implement:
- Multi-agent system with 15+ autonomous agents
- Contract Net Protocol for task negotiation and allocation
- Market-based pricing and resource coordination
- Self-organizing network adaptation mechanisms
- Emergent behavior analysis and performance metrics

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
import random
from collections import defaultdict, deque
import warnings
from enum import Enum
import time
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class Agent:
    """Autonomous agent in the ecosystem"""
    agent_id: str
    agent_type: str  # 'facility', 'transporter', 'collector', 'processor'
    location: Tuple[float, float]
    capacity: float
    current_load: float = 0.0
    resources: Dict[str, float] = field(default_factory=dict)
    capabilities: List[str] = field(default_factory=list)
    reputation_score: float = 1.0
    cost_per_unit: float = 1.0
    negotiation_history: List[Dict] = field(default_factory=list)
    
    def can_handle_task(self, task_requirements: Dict[str, Any]) -> bool:
        """Check if agent can handle a given task"""
        capacity_available = self.capacity - self.current_load >= task_requirements.get('capacity', 0)
        has_capabilities = all(cap in self.capabilities for cap in task_requirements.get('capabilities', []))
        return capacity_available and has_capabilities
    
    def calculate_bid(self, task_requirements: Dict[str, Any]) -> float:
        """Calculate bid price for a task"""
        base_cost = task_requirements.get('capacity', 0) * self.cost_per_unit
        
        # Adjust for reputation (higher reputation = premium price)
        reputation_multiplier = 1.0 + (self.reputation_score - 1.0) * 0.5
        
        # Adjust for current load (busier agents charge more)
        load_multiplier = 1.0 + (self.current_load / self.capacity) * 0.3
        
        # Market dynamics (supply/demand)
        market_multiplier = 1.0 + np.random.uniform(-0.1, 0.2)
        
        total_cost = base_cost * reputation_multiplier * load_multiplier * market_multiplier
        return total_cost
    
    def update_reputation(self, performance_score: float):
        """Update agent reputation based on performance"""
        # Exponential moving average for reputation
        alpha = 0.1
        self.reputation_score = alpha * performance_score + (1 - alpha) * self.reputation_score
        self.reputation_score = max(0.1, min(2.0, self.reputation_score))  # Bound reputation

@dataclass
class Task:
    """Task to be allocated in the ecosystem"""
    task_id: str
    task_type: str
    requirements: Dict[str, Any]
    priority: int  # 1-10, higher is more urgent
    deadline: int  # Time steps until deadline
    reward: float
    location: Tuple[float, float]
    status: str = 'pending'  # 'pending', 'assigned', 'in_progress', 'completed'
    assigned_agent: Optional[str] = None
    completion_time: Optional[int] = None

@dataclass
class ContractNetProtocol:
    """Contract Net Protocol for agent negotiations"""
    protocol_id: str
    negotiation_history: List[Dict[str, Any]] = field(default_factory=list)
    success_rate: float = 0.0
    average_negotiation_time: float = 0.0
    
    def announce_task(self, task: Task, agents: List[Agent]) -> List[Tuple[Agent, float]]:
        """Announce task to eligible agents and collect bids"""
        start_time = time.time()
        
        # Find eligible agents
        eligible_agents = [agent for agent in agents if agent.can_handle_task(task.requirements)]
        
        # Collect bids
        bids = []
        for agent in eligible_agents:
            bid_price = agent.calculate_bid(task.requirements)
            bids.append((agent, bid_price))
        
        # Record negotiation
        negotiation_time = time.time() - start_time
        self.negotiation_history.append({
            'task_id': task.task_id,
            'eligible_agents': len(eligible_agents),
            'bids_received': len(bids),
            'negotiation_time': negotiation_time
        })
        
        return bids
    
    def award_contract(self, task: Task, bids: List[Tuple[Agent, float]]) -> Optional[Agent]:
        """Award contract to best bidder"""
        if not bids:
            return None
        
        # Sort by bid price (lower is better)
        bids.sort(key=lambda x: x[1])
        
        # Select winner (consider reputation as secondary factor)
        best_agent, best_bid = bids[0]
        
        # Update statistics
        if len(self.negotiation_history) > 0:
            self.success_rate = len([h for h in self.negotiation_history if h['bids_received'] > 0]) / len(self.negotiation_history)
            self.average_negotiation_time = np.mean([h['negotiation_time'] for h in self.negotiation_history])
        
        return best_agent

@dataclass
class MarketMechanism:
    """Market-based coordination mechanism"""
    market_id: str
    price_history: Dict[str, List[float]] = field(default_factory=dict)
    supply_demand_history: Dict[str, Dict[str, List[int]]] = field(default_factory=dict)
    equilibrium_prices: Dict[str, float] = field(default_factory=dict)
    
    def update_market_prices(self, resource_type: str, supply: int, demand: int):
        """Update market prices based on supply and demand"""
        # Record supply/demand
        if resource_type not in self.supply_demand_history:
            self.supply_demand_history[resource_type] = {'supply': [], 'demand': []}
        
        self.supply_demand_history[resource_type]['supply'].append(supply)
        self.supply_demand_history[resource_type]['demand'].append(demand)
        
        # Calculate equilibrium price (simplified supply-demand model)
        if demand > 0:
            price_ratio = supply / demand
            base_price = 10.0  # Base market price
            
            if price_ratio >= 1.0:
                # Supply >= demand, price decreases
                equilibrium_price = base_price * (1.0 / (1.0 + price_ratio))
            else:
                # Demand > supply, price increases
                equilibrium_price = base_price * (2.0 - price_ratio)
            
            self.equilibrium_prices[resource_type] = equilibrium_price
            
            # Record price history
            if resource_type not in self.price_history:
                self.price_history[resource_type] = []
            self.price_history[resource_type].append(equilibrium_price)
    
    def get_market_price(self, resource_type: str) -> float:
        """Get current market price for a resource"""
        return self.equilibrium_prices.get(resource_type, 10.0)
    
    def analyze_market_dynamics(self) -> Dict[str, Any]:
        """Analyze market dynamics and trends"""
        analysis = {
            'market_volatility': {},
            'price_trends': {},
            'supply_demand_balance': {}
        }
        
        for resource_type in self.price_history:
            prices = self.price_history[resource_type]
            if len(prices) > 1:
                # Calculate volatility (standard deviation)
                analysis['market_volatility'][resource_type] = np.std(prices)
                
                # Calculate trend (last 5 vs previous 5)
                if len(prices) >= 10:
                    recent_avg = np.mean(prices[-5:])
                    previous_avg = np.mean(prices[-10:-5])
                    trend = (recent_avg - previous_avg) / previous_avg * 100
                    analysis['price_trends'][resource_type] = trend
                
                # Supply-demand balance
                if resource_type in self.supply_demand_history:
                    supply = self.supply_demand_history[resource_type]['supply'][-1]
                    demand = self.supply_demand_history[resource_type]['demand'][-1]
                    balance = supply / demand if demand > 0 else 1.0
                    analysis['supply_demand_balance'][resource_type] = balance
        
        return analysis

@dataclass
class AutonomousEcosystem:
    """Autonomous self-optimizing ecosystem"""
    ecosystem_id: str
    agents: List[Agent] = field(default_factory=list)
    tasks: List[Task] = field(default_factory=list)
    contract_protocol: ContractNetProtocol = field(default_factory=lambda: ContractNetProtocol("cnp_001"))
    market_mechanism: MarketMechanism = field(default_factory=lambda: MarketMechanism("market_001"))
    performance_history: List[Dict[str, Any]] = field(default_factory=list)
    emergent_behaviors: List[str] = field(default_factory=list)
    
    def add_agent(self, agent: Agent):
        """Add an agent to the ecosystem"""
        self.agents.append(agent)
    
    def add_task(self, task: Task):
        """Add a task to the ecosystem"""
        self.tasks.append(task)
    
    def optimize_task_allocation(self) -> Dict[str, Any]:
        """Optimize task allocation using Contract Net Protocol"""
        allocation_results = {
            'tasks_assigned': 0,
            'tasks_failed': 0,
            'total_cost': 0.0,
            'agent_utilization': {},
            'assignment_details': []
        }
        
        # Process pending tasks
        pending_tasks = [task for task in self.tasks if task.status == 'pending']
        
        for task in pending_tasks:
            # Announce task and collect bids
            bids = self.contract_protocol.announce_task(task, self.agents)
            
            # Award contract
            winning_agent = self.contract_protocol.award_contract(task, bids)
            
            if winning_agent:
                # Assign task to winning agent
                task.assigned_agent = winning_agent.agent_id
                task.status = 'assigned'
                winning_agent.current_load += task.requirements.get('capacity', 0)
                
                # Calculate cost
                task_cost = winning_agent.calculate_bid(task.requirements)
                allocation_results['total_cost'] += task_cost
                
                # Record assignment
                allocation_results['assignment_details'].append({
                    'task_id': task.task_id,
                    'agent_id': winning_agent.agent_id,
                    'cost': task_cost,
                    'agent_reputation': winning_agent.reputation_score
                })
                
                allocation_results['tasks_assigned'] += 1
                
                # Update agent utilization
                if winning_agent.agent_id not in allocation_results['agent_utilization']:
                    allocation_results['agent_utilization'][winning_agent.agent_id] = 0
                allocation_results['agent_utilization'][winning_agent.agent_id] += task.requirements.get('capacity', 0)
            else:
                allocation_results['tasks_failed'] += 1
        
        return allocation_results
    
    def simulate_ecosystem_operation(self, num_time_steps: int) -> Dict[str, Any]:
        """Simulate ecosystem operation over time"""
        simulation_results = {
            'time_steps': num_time_steps,
            'tasks_processed': 0,
            'total_reward': 0.0,
            'agent_performance': {},
            'system_efficiency': [],
            'emergence_metrics': {}
        }
        
        for time_step in range(num_time_steps):
            # Generate new tasks (randomly)
            if np.random.random() < 0.3:  # 30% chance of new task
                new_task = self._generate_random_task(time_step)
                self.add_task(new_task)
            
            # Optimize task allocation
            allocation_results = self.optimize_task_allocation()
            
            # Process assigned tasks
            for task in self.tasks:
                if task.status == 'assigned' and time_step >= task.deadline:
                    # Complete task
                    task.status = 'completed'
                    task.completion_time = time_step
                    simulation_results['tasks_processed'] += 1
                    simulation_results['total_reward'] += task.reward
                    
                    # Update agent reputation
                    if task.assigned_agent:
                        agent = next(a for a in self.agents if a.agent_id == task.assigned_agent)
                        performance_score = np.random.uniform(0.7, 1.0)  # Simulated performance
                        agent.update_reputation(performance_score)
                        
                        # Update agent performance tracking
                        if agent.agent_id not in simulation_results['agent_performance']:
                            simulation_results['agent_performance'][agent.agent_id] = {
                                'tasks_completed': 0,
                                'total_performance': 0.0
                            }
                        simulation_results['agent_performance'][agent.agent_id]['tasks_completed'] += 1
                        simulation_results['agent_performance'][agent.agent_id]['total_performance'] += performance_score
                        
                        # Release agent capacity
                        agent.current_load -= task.requirements.get('capacity', 0)
            
            # Update market prices
            total_supply = sum(agent.capacity - agent.current_load for agent in self.agents)
            total_demand = sum(task.requirements.get('capacity', 0) for task in self.tasks if task.status == 'pending')
            self.market_mechanism.update_market_prices('capacity', int(total_supply), int(total_demand))
            
            # Calculate system efficiency
            if len(self.tasks) > 0:
                efficiency = len([t for t in self.tasks if t.status == 'completed']) / len(self.tasks)
                simulation_results['system_efficiency'].append(efficiency)
            
            # Record performance history
            self.performance_history.append({
                'time_step': time_step,
                'tasks_processed': simulation_results['tasks_processed'],
                'system_efficiency': simulation_results['system_efficiency'][-1] if simulation_results['system_efficiency'] else 0,
                'market_price': self.market_mechanism.get_market_price('capacity')
            })
        
        # Analyze emergent behaviors
        simulation_results['emergence_metrics'] = self._analyze_emergent_behaviors()
        
        return simulation_results
    
    def _generate_random_task(self, time_step: int) -> Task:
        """Generate a random task"""
        task_types = ['collection', 'transport', 'processing', 'distribution']
        task_type = np.random.choice(task_types)
        
        return Task(
            task_id=f"task_{time_step}_{np.random.randint(1000)}",
            task_type=task_type,
            requirements={
                'capacity': np.random.uniform(5, 20),
                'capabilities': [task_type]
            },
            priority=np.random.randint(1, 11),
            deadline=time_step + np.random.randint(5, 15),
            reward=np.random.uniform(50, 200),
            location=(np.random.uniform(0, 100), np.random.uniform(0, 100))
        )
    
    def _analyze_emergent_behaviors(self) -> Dict[str, Any]:
        """Analyze emergent behaviors in the ecosystem"""
        emergence_metrics = {
            'agent_specialization': {},
            'reputation_distribution': {},
            'cooperation_patterns': {},
            'system_resilience': 0.0
        }
        
        # Agent specialization (which agents handle which task types)
        for agent in self.agents:
            completed_tasks = [t for t in self.tasks if t.assigned_agent == agent.agent_id and t.status == 'completed']
            if completed_tasks:
                task_types = [t.task_type for t in completed_tasks]
                emergence_metrics['agent_specialization'][agent.agent_id] = {
                    'primary_type': max(set(task_types), key=task_types.count),
                    'diversity': len(set(task_types))
                }
        
        # Reputation distribution
        reputations = [agent.reputation_score for agent in self.agents]
        emergence_metrics['reputation_distribution'] = {
            'mean': np.mean(reputations),
            'std': np.std(reputations),
            'min': np.min(reputations),
            'max': np.max(reputations)
        }
        
        # Cooperation patterns (based on contract history)
        if self.contract_protocol.negotiation_history:
            avg_bids_per_task = np.mean([h['bids_received'] for h in self.contract_protocol.negotiation_history if h['bids_received'] > 0])
            emergence_metrics['cooperation_patterns'] = {
                'avg_competition_per_task': avg_bids_per_task,
                'market_activity': len(self.contract_protocol.negotiation_history)
            }
        
        # System resilience (ability to handle task failures)
        if len(self.performance_history) > 1:
            efficiency_trend = [p['system_efficiency'] for p in self.performance_history[-10:]]
            if len(efficiency_trend) > 1:
                resilience = 1.0 - np.std(efficiency_trend)  # Lower variance = higher resilience
                emergence_metrics['system_resilience'] = max(0, resilience)
        
        return emergence_metrics
    
    def generate_ecosystem_report(self) -> Dict[str, Any]:
        """Generate comprehensive ecosystem report"""
        report = {
            'report_timestamp': time.time(),
            'ecosystem_id': self.ecosystem_id,
            'system_overview': {
                'total_agents': len(self.agents),
                'total_tasks': len(self.tasks),
                'agent_types': list(set(agent.agent_type for agent in self.agents)),
                'task_types': list(set(task.task_type for task in self.tasks))
            },
            'performance_metrics': {},
            'market_analysis': {},
            'emergence_analysis': {},
            'recommendations': []
        }
        
        # Performance metrics
        if self.performance_history:
            efficiencies = [p['system_efficiency'] for p in self.performance_history]
            report['performance_metrics'] = {
                'avg_efficiency': np.mean(efficiencies),
                'efficiency_trend': efficiencies[-1] - efficiencies[0] if len(efficiencies) > 1 else 0,
                'peak_efficiency': np.max(efficiencies),
                'efficiency_stability': 1.0 - np.std(efficiencies) if len(efficiencies) > 1 else 1.0
            }
        
        # Market analysis
        report['market_analysis'] = self.market_mechanism.analyze_market_dynamics()
        
        # Emergence analysis
        emergence_metrics = self._analyze_emergent_behaviors()
        report['emergence_analysis'] = emergence_metrics
        
        # Generate recommendations
        if report['performance_metrics'].get('avg_efficiency', 0) < 0.7:
            report['recommendations'].append('Consider adding more agents to improve task allocation efficiency')
        
        if emergence_metrics.get('system_resilience', 0) < 0.8:
            report['recommendations'].append('System resilience could be improved through better agent diversity')
        
        if len(self.agents) < 15:
            report['recommendations'].append('Consider increasing agent population for better market dynamics')
        
        return report

In [ ]:
# Initialize Autonomous Ecosystem
print("Initializing Autonomous Self-Optimizing Ecosystem...")
print("=" * 50)

# Create ecosystem
ecosystem = AutonomousEcosystem(
    ecosystem_id="autonomous_ecosystem_001"
)

# Create diverse agents
agent_configs = [
    # Facilities
    {'type': 'facility', 'count': 4, 'capacity': 50, 'capabilities': ['processing', 'storage']},
    # Transporters
    {'type': 'transporter', 'count': 6, 'capacity': 30, 'capabilities': ['transport', 'collection']},
    # Collectors
    {'type': 'collector', 'count': 5, 'capacity': 25, 'capabilities': ['collection', 'sorting']},
    # Processors
    {'type': 'processor', 'count': 3, 'capacity': 40, 'capabilities': ['processing', 'distribution']}
]

agent_id_counter = 0
for config in agent_configs:
    for i in range(config['count']):
        agent_id_counter += 1
        agent = Agent(
            agent_id=f"{config['type']}_{agent_id_counter:03d}",
            agent_type=config['type'],
            location=(np.random.uniform(0, 100), np.random.uniform(0, 100)),
            capacity=config['capacity'],
            capabilities=config['capabilities'],
            cost_per_unit=np.random.uniform(0.8, 1.5),
            reputation_score=np.random.uniform(0.8, 1.2)
        )
        ecosystem.add_agent(agent)

print(f"Ecosystem Created: {ecosystem.ecosystem_id}")
print(f"Total Agents: {len(ecosystem.agents)}")

# Display agent distribution
agent_types = defaultdict(int)
for agent in ecosystem.agents:
    agent_types[agent.agent_type] += 1

print(f"\nAgent Distribution:")
for agent_type, count in agent_types.items():
    print(f"  {agent_type.capitalize()}: {count}")

# Display agent capabilities
print(f"\nAgent Capabilities:")
capabilities_summary = defaultdict(list)
for agent in ecosystem.agents:
    for capability in agent.capabilities:
        capabilities_summary[capability].append(agent.agent_type)

for capability, agent_types_list in capabilities_summary.items():
    unique_types = list(set(agent_types_list))
    print(f"  {capability.capitalize()}: {', '.join(unique_types)}")

# Display ecosystem characteristics
print(f"\nEcosystem Characteristics:")
total_capacity = sum(agent.capacity for agent in ecosystem.agents)
avg_reputation = np.mean([agent.reputation_score for agent in ecosystem.agents])
avg_cost = np.mean([agent.cost_per_unit for agent in ecosystem.agents])

print(f"  Total System Capacity: {total_capacity:.1f}")
print(f"  Average Agent Reputation: {avg_reputation:.3f}")
print(f"  Average Cost per Unit: {avg_cost:.3f}")
print(f"  Contract Net Protocol: {ecosystem.contract_protocol.protocol_id}")
print(f"  Market Mechanism: {ecosystem.market_mechanism.market_id}")
print(f"  System Ready: True")

# Generate initial tasks
print(f"\nGenerating Initial Tasks...")

initial_tasks = 15
for i in range(initial_tasks):
    task = ecosystem._generate_random_task(0)
    ecosystem.add_task(task)

print(f"Initial Tasks Generated: {initial_tasks}")
print(f"Task Types: {list(set(task.task_type for task in ecosystem.tasks))}")

# Display task requirements
print(f"\nTask Requirements Analysis:")
task_types = defaultdict(list)
for task in ecosystem.tasks:
    task_types[task.task_type].append(task.requirements.get('capacity', 0))

for task_type, capacities in task_types.items():
    avg_capacity = np.mean(capacities)
    print(f"  {task_type.capitalize()}: {len(capacities)} tasks, avg capacity {avg_capacity:.1f}")

print(f"\nSystem Initialization Complete!")
print(f"Ready for autonomous ecosystem simulation...")

Initializing Autonomous Self-Optimizing Ecosystem...
Epoch 60: Loss=0.3844, Val_Loss=1.7658, Robustness=0.885
Decision 20: Demand Allocation
  Decision Quality: 0.757
  Trust Score: 0.971
  Human Weight: 0.24, AI Weight: 0.76
  Success: Yes


In [ ]:
try:
    def run_ecosystem_simulation(ecosystem: AutonomousEcosystem, num_time_steps: int = 50) -> Dict[str, Any]:
        """Run comprehensive ecosystem simulation"""
        
        print(f"Running Autonomous Ecosystem Simulation...")
        print(f"Time Steps: {num_time_steps}")
        print("=" * 40)
        
        # Run simulation
        simulation_results = ecosystem.simulate_ecosystem_operation(num_time_steps)
        
        # Display results
        print(f"\nSimulation Results:")
        print(f"  Tasks Processed: {simulation_results['tasks_processed']}")
        print(f"  Total Reward: {simulation_results['total_reward']:.2f}")
        print(f"  Time Steps: {simulation_results['time_steps']}")
        
        # Agent performance
        print(f"\nAgent Performance:")
        for agent_id, performance in simulation_results['agent_performance'].items():
            avg_performance = performance['total_performance'] / performance['tasks_completed']
            print(f"  {agent_id}: {performance['tasks_completed']} tasks, avg performance {avg_performance:.3f}")
        
        # System efficiency
        if simulation_results['system_efficiency']:
            final_efficiency = simulation_results['system_efficiency'][-1]
            avg_efficiency = np.mean(simulation_results['system_efficiency'])
            print(f"\nSystem Efficiency:")
            print(f"  Final Efficiency: {final_efficiency:.3f}")
            print(f"  Average Efficiency: {avg_efficiency:.3f}")
            print(f"  Peak Efficiency: {np.max(simulation_results['system_efficiency']):.3f}")
        
        # Emergence metrics
        emergence = simulation_results['emergence_metrics']
        print(f"\nEmergent Behaviors:")
        
        if 'agent_specialization' in emergence:
            print(f"  Agent Specialization:")
            for agent_id, specialization in emergence['agent_specialization'].items():
                print(f"    {agent_id}: {specialization['primary_type']} (diversity: {specialization['diversity']})")
        
        if 'reputation_distribution' in emergence:
            rep_dist = emergence['reputation_distribution']
            print(f"  Reputation Distribution: mean={rep_dist['mean']:.3f}, std={rep_dist['std']:.3f}")
            print(f"    Range: [{rep_dist['min']:.3f}, {rep_dist['max']:.3f}]")
        
        if 'cooperation_patterns' in emergence:
            cooperation = emergence['cooperation_patterns']
            print(f"  Cooperation Patterns:")
            print(f"    Avg Competition per Task: {cooperation.get('avg_competition_per_task', 0):.1f}")
            print(f"    Market Activity: {cooperation.get('market_activity', 0)} negotiations")
        
        print(f"  System Resilience: {emergence.get('system_resilience', 0):.3f}")
        
        # Contract Net Protocol performance
        print(f"\nContract Net Protocol Performance:")
        print(f"  Success Rate: {ecosystem.contract_protocol.success_rate:.3f}")
        print(f"  Avg Negotiation Time: {ecosystem.contract_protocol.average_negotiation_time:.4f}s")
        print(f"  Total Negotiations: {len(ecosystem.contract_protocol.negotiation_history)}")
        
        # Market analysis
        market_analysis = ecosystem.market_mechanism.analyze_market_dynamics()
        print(f"\nMarket Dynamics:")
        
        if market_analysis.get('market_volatility'):
            for resource, volatility in market_analysis['market_volatility'].items():
                print(f"  {resource.capitalize()} Volatility: {volatility:.3f}")
        
        if market_analysis.get('price_trends'):
            for resource, trend in market_analysis['price_trends'].items():
                direction = "increasing" if trend > 0 else "decreasing"
                print(f"  {resource.capitalize()} Price Trend: {direction} ({trend:+.1f}%)")
        
        if market_analysis.get('supply_demand_balance'):
            for resource, balance in market_analysis['supply_demand_balance'].items():
                status = "surplus" if balance > 1.0 else "shortage" if balance < 1.0 else "balanced"
                print(f"  {resource.capitalize()} Market: {status} (ratio: {balance:.2f})")
        
        return simulation_results
    
    # Run the simulation
    simulation_results = run_ecosystem_simulation(ecosystem, num_time_steps=50)
    
    # Generate comprehensive report
    final_report = ecosystem.generate_ecosystem_report()
    
    print(f"\nFinal Ecosystem Report:")
    print("=" * 30)
    
    overview = final_report['system_overview']
    print(f"System Overview:")
    print(f"  Total Agents: {overview['total_agents']}")
    print(f"  Total Tasks: {overview['total_tasks']}")
    print(f"  Agent Types: {', '.join(overview['agent_types'])}")
    print(f"  Task Types: {', '.join(overview['task_types'])}")
    
    if 'performance_metrics' in final_report and final_report['performance_metrics']:
        perf = final_report['performance_metrics']
        print(f"\nPerformance Metrics:")
        print(f"  Average Efficiency: {perf['avg_efficiency']:.3f}")
        print(f"  Efficiency Trend: {perf['efficiency_trend']:+.3f}")
        print(f"  Peak Efficiency: {perf['peak_efficiency']:.3f}")
        print(f"  Efficiency Stability: {perf['efficiency_stability']:.3f}")
    
    if 'emergence_analysis' in final_report and final_report['emergence_analysis']:
        emergence = final_report['emergence_analysis']
        print(f"\nEmergence Analysis:")
        if 'system_resilience' in emergence:
            print(f"  System Resilience: {emergence['system_resilience']:.3f}")
        if 'reputation_distribution' in emergence:
            rep_dist = emergence['reputation_distribution']
            print(f"  Reputation Range: [{rep_dist['min']:.3f}, {rep_dist['max']:.3f}]")
    
    print(f"\nRecommendations:")
    for i, recommendation in enumerate(final_report['recommendations'], 1):
        print(f"  {i}. {recommendation}")
    
    # Performance assessment
    print(f"\nPerformance Assessment:")
    avg_efficiency = final_report['performance_metrics'].get('avg_efficiency', 0)
    system_resilience = final_report['emergence_analysis'].get('system_resilience', 0)
    success_rate = ecosystem.contract_protocol.success_rate
    
    if avg_efficiency > 0.8:
        print(f"  System Efficiency: EXCELLENT")
    elif avg_efficiency > 0.6:
        print(f"  System Efficiency: GOOD")
    else:
        print(f"  System Efficiency: NEEDS IMPROVEMENT")
    
    if system_resilience > 0.8:
        print(f"  System Resilience: EXCELLENT")
    elif system_resilience > 0.6:
        print(f"  System Resilience: GOOD")
    else:
        print(f"  System Resilience: NEEDS IMPROVEMENT")
    
    if success_rate > 0.9:
        print(f"  Task Allocation: EXCELLENT")
    elif success_rate > 0.7:
        print(f"  Task Allocation: GOOD")
    else:
        print(f"  Task Allocation: NEEDS IMPROVEMENT")
    
    # Overall assessment
    overall_score = (avg_efficiency + system_resilience + success_rate) / 3
    print(f"\nOverall Ecosystem Score: {overall_score:.3f}")
    
    if overall_score > 0.8:
        print(f"  Overall Status: EXCELLENT - Autonomous ecosystem performing optimally")
    elif overall_score > 0.6:
        print(f"  Overall Status: GOOD - Autonomous ecosystem functioning well")
    else:
        print(f"  Overall Status: NEEDS IMPROVEMENT - Autonomous ecosystem requires optimization")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    def visualize_ecosystem_performance(ecosystem: AutonomousEcosystem, 
                                       simulation_results: Dict[str, Any]):
        """Visualize ecosystem performance and emergent behaviors"""
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        # Plot 1: System Efficiency Over Time
        ax1.set_title('System Efficiency Over Time', fontsize=14, fontweight='bold')
        
        if ecosystem.performance_history:
            time_steps = [p['time_step'] for p in ecosystem.performance_history]
            efficiencies = [p['system_efficiency'] for p in ecosystem.performance_history]
            
            ax1.plot(time_steps, efficiencies, 'b-', linewidth=2, marker='o', markersize=3)
            ax1.set_xlabel('Time Step')
            ax1.set_ylabel('System Efficiency')
            ax1.set_ylim(0, 1)
            ax1.grid(True, alpha=0.3)
            
            # Add trend line
            if len(efficiencies) > 1:
                z = np.polyfit(time_steps, efficiencies, 1)
                p = np.poly1d(z)
                ax1.plot(time_steps, p(time_steps), "r--", alpha=0.8, label='Trend')
                ax1.legend()
        
        # Plot 2: Agent Reputation Distribution
        ax2.set_title('Agent Reputation Distribution', fontsize=14, fontweight='bold')
        
        reputations = [agent.reputation_score for agent in ecosystem.agents]
        agent_types = [agent.agent_type for agent in ecosystem.agents]
        
        # Create box plot by agent type
        type_reputations = {}
        for agent_type in set(agent_types):
            type_reputations[agent_type] = [rep for rep, atype in zip(reputations, agent_types) if atype == agent_type]
        
        ax2.boxplot(type_reputations.values(), labels=type_reputations.keys())
        ax2.set_ylabel('Reputation Score')
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Market Price Dynamics
        ax3.set_title('Market Price Dynamics', fontsize=14, fontweight='bold')
        
        if ecosystem.market_mechanism.price_history:
            for resource_type, prices in ecosystem.market_mechanism.price_history.items():
                time_points = list(range(len(prices)))
                ax3.plot(time_points, prices, linewidth=2, marker='o', markersize=3, label=resource_type.capitalize())
            
            ax3.set_xlabel('Time Step')
            ax3.set_ylabel('Market Price')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        
        # Plot 4: Agent Specialization Heatmap
        ax4.set_title('Agent Task Specialization', fontsize=14, fontweight='bold')
        
        emergence = simulation_results['emergence_metrics']
        if 'agent_specialization' in emergence:
            specializations = emergence['agent_specialization']
            
            # Create specialization matrix
            agent_ids = list(specializations.keys())
            task_types = list(set(spec['primary_type'] for spec in specializations.values()))
            
            specialization_matrix = np.zeros((len(agent_ids), len(task_types)))
            
            for i, agent_id in enumerate(agent_ids):
                spec = specializations[agent_id]
                primary_type = spec['primary_type']
                diversity = spec['diversity']
                
                if primary_type in task_types:
                    j = task_types.index(primary_type)
                    # Higher value for more specialized agents (lower diversity)
                    specialization_matrix[i, j] = 1.0 / diversity
            
            im = ax4.imshow(specialization_matrix, cmap='YlOrRd', aspect='auto')
            ax4.set_xticks(range(len(task_types)))
            ax4.set_xticklabels(task_types, rotation=45)
            ax4.set_yticks(range(len(agent_ids)))
            ax4.set_yticklabels([aid.split('_')[0] for aid in agent_ids])
            ax4.set_xlabel('Task Type')
            ax4.set_ylabel('Agent ID')
            
            # Add colorbar
            plt.colorbar(im, ax=ax4, label='Specialization Level')
        
        plt.tight_layout()
        plt.show()
        
        # Additional analysis
        print("\nEcosystem Performance Analysis:")
        print("=" * 50)
        
        # Contract Net Protocol analysis
        if ecosystem.contract_protocol.negotiation_history:
            negotiations = ecosystem.contract_protocol.negotiation_history
            avg_bids = np.mean([n['bids_received'] for n in negotiations if n['bids_received'] > 0])
            avg_time = np.mean([n['negotiation_time'] for n in negotiations])
            
            print(f"Contract Net Protocol Analysis:")
            print(f"  Total Negotiations: {len(negotiations)}")
            print(f"  Average Bids per Task: {avg_bids:.1f}")
            print(f"  Average Negotiation Time: {avg_time:.4f}s")
            print(f"  Success Rate: {ecosystem.contract_protocol.success_rate:.3f}")
        
        # Market efficiency analysis
        market_analysis = ecosystem.market_mechanism.analyze_market_dynamics()
        print(f"\nMarket Efficiency Analysis:")
        
        if market_analysis.get('market_volatility'):
            avg_volatility = np.mean(list(market_analysis['market_volatility'].values()))
            print(f"  Average Market Volatility: {avg_volatility:.3f}")
            
            if avg_volatility < 0.5:
                print(f"  Market Stability: EXCELLENT")
            elif avg_volatility < 1.0:
                print(f"  Market Stability: GOOD")
            else:
                print(f"  Market Stability: VOLATILE")
        
        # Agent performance analysis
        print(f"\nAgent Performance Analysis:")
        
        # Top performing agents
        agent_performances = []
        for agent in ecosystem.agents:
            completed_tasks = [t for t in ecosystem.tasks if t.assigned_agent == agent.agent_id and t.status == 'completed']
            if completed_tasks:
                agent_performances.append((agent.agent_id, len(completed_tasks), agent.reputation_score))
        
        if agent_performances:
            agent_performances.sort(key=lambda x: x[1], reverse=True)
            print(f"  Top 5 Agents by Tasks Completed:")
            for i, (agent_id, tasks, reputation) in enumerate(agent_performances[:5], 1):
                print(f"    {i}. {agent_id}: {tasks} tasks (reputation: {reputation:.3f})")
        
        # Emergent behavior analysis
        emergence = simulation_results['emergence_metrics']
        print(f"\nEmergent Behavior Analysis:")
        
        if 'system_resilience' in emergence:
            resilience = emergence['system_resilience']
            print(f"  System Resilience: {resilience:.3f}")
            
            if resilience > 0.8:
                print(f"  Resilience Assessment: EXCELLENT - Highly adaptive system")
            elif resilience > 0.6:
                print(f"  Resilience Assessment: GOOD - Moderately adaptive system")
            else:
                print(f"  Resilience Assessment: NEEDS IMPROVEMENT - Low adaptability")
        
        if 'cooperation_patterns' in emergence:
            cooperation = emergence['cooperation_patterns']
            competition = cooperation.get('avg_competition_per_task', 0)
            print(f"  Market Competition: {competition:.1f} bids per task")
            
            if competition > 3.0:
                print(f"  Competition Level: HIGH - Healthy market dynamics")
            elif competition > 1.5:
                print(f"  Competition Level: MODERATE - Balanced market")
            else:
                print(f"  Competition Level: LOW - Need more agents")
        
        # System optimization recommendations
        print(f"\nSystem Optimization Recommendations:")
        
        if ecosystem.contract_protocol.success_rate < 0.8:
            print(f"  1. Improve agent diversity to increase task allocation success")
        
        if len(ecosystem.agents) < 20:
            print(f"  2. Add more agents to improve market competition")
        
        if emergence.get('system_resilience', 0) < 0.7:
            print(f"  3. Enhance agent adaptability for better resilience")
        
        if market_analysis.get('market_volatility') and np.mean(list(market_analysis['market_volatility'].values())) > 1.0:
            print(f"  4. Implement price stabilization mechanisms")
        
        print(f"  5. Continuously monitor emergent behaviors for system improvements")
        
        return simulation_results
    
    # Visualize ecosystem performance
    visualize_ecosystem_performance(ecosystem, simulation_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'simulation_results' is not defined...]

### Why This Tier Exists vs Previous Tiers
The Autonomous Self-Optimizing Ecosystem addresses fundamental limitations of all previous approaches:
- **Emergent Intelligence**: Complex behaviors emerge from simple agent rules
- **Self-Organization**: System adapts without centralized control
- **Market Efficiency**: Resource allocation through market mechanisms
- **Scalability**: Easily add/remove agents without system redesign
- **Resilience**: Distributed system robust to failures
- **Autonomy**: Minimal human intervention required

### Pros vs Cons
**Advantages:**
- **Emergence**: Complex behaviors emerge from simple agent rules
- **Autonomy**: Reduced need for human intervention and management
- **Flexibility**: Easy to add/remove agents without system redesign
- **Scalability**: System scales naturally with agent population
- **Resilience**: Distributed architecture robust to failures
- **Market Efficiency**: Optimal resource allocation through pricing
- **Adaptability**: System adapts to changing conditions

**Disadvantages:**
- **Complexity**: Difficult to predict and control emergent behaviors
- **Coordination Overhead**: Communication and negotiation between agents
- **Initial Setup**: Requires careful agent design and parameter tuning
- **Debugging Challenges**: Hard to trace issues in distributed systems
- **Performance Variability**: Results may vary due to agent interactions
- **Resource Requirements**: Computational overhead for agent coordination
- **Predictability**: Less predictable than centralized optimization approaches

### When to Use This Tier
- **Large-Scale Networks** with many nodes and complex interactions
- **Dynamic Environments** requiring continuous adaptation
- **Distributed Operations** across multiple locations
- **Resource Allocation** problems with market dynamics
- **Resilient Systems** requiring fault tolerance
- **Autonomous Operations** minimizing human intervention
- **Complex Supply Chains** with many stakeholders

### Key Insights from This Analysis
1. **Emergent Intelligence** of 15+ autonomous agents
2. **Contract Net Protocol** for efficient task allocation
3. **Market-Based Coordination** with dynamic pricing
4. **Self-Organization** and adaptation capabilities
5. **System Resilience** through distributed architecture
6. **Performance Optimization** through autonomous learning

### Autonomous Ecosystem Characteristics
- **Multi-Agent Architecture** with diverse agent types
- **Contract Net Protocol** for task negotiation and allocation
- **Market-Based Pricing** for resource coordination
- **Self-Organizing Network** adaptation mechanisms
- **Emergent Behavior Analysis** and performance metrics
- **System Resilience** and autonomous optimization